# 4. Geospatial processing

### [read] Summarise data availability of addresses among the fixed_table
- So we know what we're working with in terms of different types

In [1]:
# ### [read] Summarise data availability of addresses among the fixed_table
# - So we know what we're working with in terms of different types
# This should be lat/long, full address, address decomposed across the different fields, just postcode, the nothing
# And if its primary trading address or registered office address
import ibis

address_hierachy = [
    {'primary_trading_address_latitude', 'primary_trading_address_longitude'},
    {'primary_trading_address'},
    {'ro_latitude', 'ro_longitude'},
    {'ro_address'},
    {'ro_address_line_1', 'ro_full_postcode'},
    {'ro_full_postcode'},
    {'ro_postcode'}
]

# Dynamically builds an ibis.cases() expression from a list of column sets.
def build_location_source_cases(table: ibis.expr.types.Table, hierarchy: list[set]):
    cases_list = []
    
    for i, field_set in enumerate(hierarchy, start=1):
        # 1. Build the logical condition: EVERY column in the set must be not-null
        condition = None
        for col_name in field_set:
            is_not_null = table[col_name].notnull()
            condition = is_not_null if condition is None else condition & is_not_null
        
        # 2. Store as a (condition, result_value) tuple
        cases_list.append((condition, i))
        
    # 3. Unpack the list of tuples into ibis.cases. 
    # Fallback MUST be an integer to match the column type.
    fallback_lvl = len(hierarchy) + 1
    return ibis.cases(*cases_list, else_=fallback_lvl)

### [write] Create new column for address geocoding

In [ ]:
import ibis
from utils.f_0_dirs import get_data_dirs
from f_3_spatial import convert_dms_to_decimal

old_table_name = "fame_fixed_filtered"
new_table_name = "working_fixed"

# Initialize connection
dirs = get_data_dirs()
con = ibis.duckdb.connect(str(dirs.db_path))
con.raw_sql("INSTALL spatial; LOAD spatial;")

# Reference the existing tables
fame_fixed = con.table(old_table_name)

# 1. Parse availability into an indicator and extract the best available text address
working_raw = fame_fixed.mutate(
    address_raw_lvl = build_location_source_cases(fame_fixed, address_hierachy),
    address_raw = ibis.coalesce(
        fame_fixed.primary_trading_address,
        fame_fixed.ro_address,
        # Fallback: concatenate the separate lines and postcode if above are null
        ibis.literal(", ").join(
            ibis.array([
                fame_fixed.ro_address_line_1, 
                fame_fixed.ro_address_line_2, 
                fame_fixed.ro_full_postcode
            ]).filter(lambda x: x.notnull())
        ),
        fame_fixed.ro_full_postcode,
        fame_fixed.ro_postcode
    )
)

# 2. Extract and clean postcodes for Levels 2, 4, 5, 6, and 7
# Regex matches standard UK postcode formats. We strip whitespace and uppercase for perfect joins.
uk_postcode_regex = r"([A-Za-z]{1,2}\d[A-Za-z\d]?\s?\d[A-Za-z]{2})"
working_pc = working_raw.mutate(
    extracted_postcode = ibis.cases(
        (working_raw.address_raw_lvl.isin([1, 2]), working_raw.primary_trading_address.re_extract(uk_postcode_regex, 1)),
        (working_raw.address_raw_lvl.isin([3, 4]), working_raw.ro_address.re_extract(uk_postcode_regex, 1)),
        (working_raw.address_raw_lvl.isin([5, 6]), working_raw.ro_full_postcode),
        (working_raw.address_raw_lvl == 7, working_raw.ro_postcode),
        else_=ibis.literal(None, type="string")
    ).upper().re_replace(r"\s+", "") 
)

# Clean the ONS directory postcodes using the same logic for the join
# ons_table_name = "ons_postcode_lookup" # Assume this exists with 'postcode', 'lat', 'lon'
# ons_lookup = con.table(ons_table_name) 
# ons_lookup_clean = ons_lookup.mutate(
#     join_pc = ons_lookup.postcode.upper().re_replace(r"\s+", "")
# )

# # 3. Join firm data with the ONS lookup
# working_joined = working_fixed_pc.left_join(
#     ons_lookup_clean,
#     working_fixed_pc.extracted_postcode == ons_lookup_clean.join_pc
# )

# 4. Extract standardized spatial coordinates based on ALL 7 levels of the hierarchy
working_fixed = working_pc.mutate(
    address_case = ibis.cases(
        (working_pc.address_raw_lvl <= 2, ibis.literal('pta')),
        (working_pc.address_raw_lvl <= 7, ibis.literal('ro')),
        else_=ibis.literal(None, type="string")
    ),
    address_lvl = ibis.cases(
        (working_pc.address_raw_lvl <= 2, working_pc.address_raw_lvl),
        (working_pc.address_raw_lvl <= 7, working_pc.address_raw_lvl - 2),
        else_=ibis.literal(None, type="int64")
    ),
    lat_dec = ibis.cases(
        (working_pc.address_raw_lvl == 1, convert_dms_to_decimal(working_pc.primary_trading_address_latitude)),
        (working_pc.address_raw_lvl == 3, convert_dms_to_decimal(working_pc.ro_latitude)),
        else_=ibis.literal(None, type='float64') # working_pc.lat # Automatically covers levels 2, 4, 5, 6, and 7
    ),
    lon_dec = ibis.cases(
        (working_pc.address_raw_lvl == 1, convert_dms_to_decimal(working_pc.primary_trading_address_longitude)),
        (working_pc.address_raw_lvl == 3, convert_dms_to_decimal(working_pc.ro_longitude)),
        else_=ibis.literal(None, type='float64') # working_pc.lon # Automatically covers levels 2, 4, 5, 6, and 7
    )
).select('registered_number', 'address_raw_lvl', 'address_raw', 'extracted_postcode',
         'address_case', 'address_lvl', 'lat_dec', 'lon_dec')
#.drop("extracted_postcode", "join_pc", "postcode", "lat", "lon") # Drop join artifacts

row_count = working_fixed.count().execute()
display(working_fixed.sample(30 / row_count).execute())

❌ DATA_DIR path does not exist or is not a directory: /mnt/h/Other computers/My computer/fame_clean/1_FAME_raw_data/2025.07.30


,registered_number,address_raw_lvl,address_raw,extracted_postcode,address_case,address_lvl,lat_dec,lon_dec
0,11166417,3,"Suite 3, Regency House, 91 Western Road, Brigh...",BN12NW,ro,1,50.824389,-0.149750
1,08782065,1,"66 City Road, London, London, EC1Y 2AL",EC1Y2AL,pta,1,51.517472,-0.133167
2,SC263289,3,"Saltire Court 20 Castle Terrace, Edinburgh, Mi...",EH12EG,ro,1,55.947972,-3.204222
3,07740792,2,"Field Court C of E Infant Academ, Courtfield R...",GL24UF,pta,2,NaN,NaN
4,NI068591,1,"4th Floor Craig Plaza, 51 Fountain Street, Bel...",BT15EA,pta,1,54.598528,-5.927889
5,08849026,2,"c/o Corporation Service Company, 5 Churchill P...",E145HU,pta,2,NaN,NaN
6,07048979,2,"Macron Stadium Burnden Way, Lostock, Bolton, L...",BL66JW,pta,2,NaN,NaN
7,05391898,3,"2 Minster Court, London, EC3R 7BB",EC3R7BB,ro,1,51.511389,-0.080972
8,06929208,3,"30 Finsbury Square, London, EC2A 1AG",EC2A1AG,ro,1,51.521000,-0.085722
9,09949360,2,"7 Treadaway Business Centre, Treadaway Hill, L...",HP109RS,pta,2,NaN,NaN


In [ ]:
print(f"✅ Inserting columns into new '{new_table_name}' table.")
con.create_table(new_table_name, working_fixed, overwrite=True)

# Verify the final materialized table
final_table = con.table(new_table_name)
row_count = final_table.count().execute()
col_count = len(final_table.columns)

print(f"✅ Materialized '{new_table_name}' table.")
print(f"📊 Number of rows: {row_count:,}")
print(f"📊 Number of columns: {col_count}")
print(f"\nSample of {new_table_name}:")
display(final_table.sample(10 / row_count).execute())